# March Mania · Frozen-early-rating temporal form
**Feature investigation 06 · different signal, fixed learner, bounded comparisons**

The schedule-record representation did not replicate: the three additional seasons averaged **+0.0002388 Brier** and only one improved. Its expansion and component ablations remain stopped. The earlier shooting additions are not included either.

This notebook engineers **four temporal features** and compares the same 16-input reference against two new two-feature families and their combination. Ratings used to define surprises are frozen at **day 100**; later game results do not enter those rating fits. Existing full-season reference snapshots remain unchanged.

**Two already-consumed historical seasons, 2017 and 2019. Not untouched tests or leaderboard results.** Maximum new work: 14 regular-season rating fits and 13 tournament-classifier fits. Three existing reference classifiers are replayed. No Git/AWS API writes, external downloads, automatic full-bank rebuild, or submissions.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / 'run_round06.py').is_file():
    KIT = Path.home() / 'march_temporal_form'
assert (KIT / 'run_round06.py').is_file(), 'Open the notebook from march_temporal_form.'
sys.path.insert(0, str(KIT))
from run_round06 import run_stage
from form_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Output folder:', KIT / 'private_runs')
print('Keep previous research folders intact. No environment reinstall is required.')

## 1 · Preserve the failed replication
Negative delta is better. Including the season that selected the hypothesis disguises its lack of replication. Do not lower the old gate to force expansion.

In [ ]:
previous = pd.read_csv(KIT / 'evidence' / 'replication_metrics.csv')
display(previous.loc[previous.recipe.eq('anchor_record'), ['Gender','Season','brier','anchor_brier','delta_vs_anchor','role']].round(7))
print(json.dumps(json.loads((KIT / 'evidence' / 'gate.json').read_text()), indent=2))

## 2 · Build the new representation
Fit one additive offense/defense ridge model per population-season using clean regular-season games through day 100. Freeze its team coefficients, average efficiency and venue coefficient. Score days 101–132 against that expectation.

**Late level (two features):** shrunk recency-weighted offense and defense surprises, with a fixed 14-day half-life.

**Late change (two features):** shrunk surprise in days 117–132 minus days 101–116, separately for offense and defense. Each mean has five zero-centered pseudo-games. Unknown early opponents are excluded and counted; absent support is not interpreted as observed zero performance.

These are hypotheses about change, not proven momentum, injury measurements, or a reproduction of KenPom. Opponents can change too, making frozen expectations stale. A different temporal representation must earn its place through the measured comparisons.

Preparation ceiling: **300 seconds**. One checkpoint per early rating and one per completed team-season feature snapshot. It reuses 14 full-season base snapshots.

In [ ]:
run_stage('prepare', max_seconds=300)
RUN = Path(json.loads((KIT / 'reports' / 'latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'feature_registry.csv').query('new_candidate == True')[['feature','family','description']])

## 3 · Inspect source availability before interpreting features
Coverage and exclusions are data-quality evidence, not feature selection using validation labels. A tournament participant with no usable early or late support causes a stop instead of a fabricated rating. Missing support in one late subwindow is documented and shrunk to the declared prior.

In [ ]:
display(pd.read_csv(RUN / 'coverage.csv'))
print('Early ratings use day <= 100; residual games use 100 < day <= 132.')

## 4 · Fixed-recipe comparisons on two historical seasons
| Configuration | Inputs |
|---|---:|
| Reference | 16 |
| Reference + late level | 18 |
| Reference + late change | 18 |
| Reference + both | 20 |

Separate men/women, validation years 2017 and 2019, tournament training starts in 2013 and ends the year before each fold. Main draw only. Fixed logistic C=0.1, training-only RMS scaling, mirrored orientations and total physical-game weight one. Only training-constant columns can be removed.

Sixteen comparisons: **13 new classifier fits**, **3 existing baseline replays**. No tuning, new calibration, ensemble, shooting/record additions, or global feature screening. Evaluation ceiling: **240 seconds**.

In [ ]:
run_stage('evaluate', max_seconds=240)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Gender','Season','recipe','games','train_games','brier','delta_vs_anchor','source']].round(7))
display(pd.read_csv(RUN / 'ablations.csv').round(7))

## 5 · Decision and training-only redundancy
A mean delta at most −0.0005 with improvement in **both** seasons is only a reason to consider unchanged broader replication. Nothing is promoted automatically. Two consumed seasons and multiple comparisons are not proof of generalization. Family additions conditional on the other family also need examination.

The current submitted model differs from this fixed learner; a positive result would still need broader validation and stronger-production-recipe transfer.

In [ ]:
print(json.dumps(json.loads((RUN / 'decisions.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'training_redundancy.csv').round(5))
display(pd.read_csv(RUN / 'coefficients.csv').round(5))

## 6 · Interactive evidence
Ten Plotly charts. Sample-support and calibration plots are descriptive. Hover over points for team IDs; no outcome-specific override rules should be derived from individual errors.

In [ ]:
plots = figures(RUN, KIT / 'evidence')
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 7 · Preserve and return
Reporting ceiling **120 seconds**. The return ZIP excludes raw game rows, fitted models, per-game predictions and edited repository notebooks. The standalone HTML contains the ten charts. Save this notebook with **Ctrl+S**, download `reports/milestone_06_return.zip`, and stop this milestone here. Do not restart earlier failed feature expansions.

In [ ]:
run_stage('report', max_seconds=120)
receipt = json.loads((KIT / 'reports' / 'latest_report.json').read_text())
print('Return archive:', receipt['return_zip'])
print('Interactive HTML:', receipt['html'])
display(FileLink(str(Path(receipt['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(receipt['html']).relative_to(KIT))))
print('No new leaderboard score or automatic feature promotion. Keep private_runs for resumption.')